# ORION — hierarchical-memory regime measurement (Kaggle-ready)

Self-contained. **No internet or repo clone required.** It measures the two
ORION ratios on whatever accelerator this Kaggle session has:

- **R_C** = fast-memory residency (C_fast / working-set)
- **R_B** = compute–transfer overlap (T_comp / T_transfer)

**How to run on Kaggle**
1. *Settings → Accelerator →* pick **GPU T4 x2** or **GPU P100** (or **TPU VM v3-8**).
2. *Run All*. Everything is timed with CUDA events on GPU (real measurement).
3. Copy the printed **`[paper paste-back]`** block and the saved
   `orion_accel_<device>.json` from the **Output** panel.

Torch is pre-installed on Kaggle GPU images; if torch is missing the notebook
falls back to a NumPy/CPU measurement so it still runs everywhere.


In [ ]:
# === ORION measurement engine (self-contained) ===============================
import json, math, os, platform, statistics, time
from pathlib import Path

NAN = float("nan")
SEED = 20260818


class NumpyBackend:
    name, device = "numpy", "CPU(numpy)"

    def __init__(self):
        import numpy as np
        self.np = np
        self.rng = np.random.default_rng(SEED)

    def new_weight(self, d):
        return (self.rng.standard_normal((d, d), dtype="float32")
                / self.np.float32(d ** 0.5))

    new_host_weight = new_weight

    def transfer_in(self, hw):
        t0 = time.perf_counter(); dev = hw.copy()
        return dev, time.perf_counter() - t0

    def matmul(self, x, w, r):
        t0 = time.perf_counter(); y = x
        for _ in range(r):
            y = self.np.matmul(y, w)
        if y.shape[0] < 0:
            raise RuntimeError
        return y, time.perf_counter() - t0

    def new_activation(self, b, d):
        return self.rng.standard_normal((b, d), dtype="float32")


class TorchBackend:
    name = "torch"

    def __init__(self, want_tpu=False):
        import torch
        self.torch = torch
        self.is_xla = False
        if want_tpu:
            import torch_xla.core.xla_model as xm  # noqa
            self.xm = xm
            self.dev = xm.xla_device()
            self.is_xla = True
            self.device = "TPU(XLA)"
        elif torch.cuda.is_available():
            self.dev = torch.device("cuda")
            self.device = f"GPU:{torch.cuda.get_device_name(0)}"
        else:
            self.dev = torch.device("cpu")
            self.device = "CPU(torch)"
        torch.manual_seed(SEED)

    def _sync(self):
        if self.is_xla:
            self.xm.mark_step(); self.xm.wait_device_ops()
        elif self.dev.type == "cuda":
            self.torch.cuda.synchronize()

    def new_weight(self, d):
        return self.torch.randn(d, d, device=self.dev,
                                dtype=self.torch.float32) / (d ** 0.5)

    def new_host_weight(self, d):
        w = self.torch.randn(d, d, dtype=self.torch.float32) / (d ** 0.5)
        if self.dev.type == "cuda":
            w = w.pin_memory()
        return w

    def transfer_in(self, hw):
        t = self.torch
        if self.dev.type == "cuda":
            s = t.cuda.Event(enable_timing=True); e = t.cuda.Event(enable_timing=True)
            t.cuda.synchronize(); s.record()
            dev = hw.to(self.dev, non_blocking=True)
            e.record(); t.cuda.synchronize()
            return dev, s.elapsed_time(e) / 1e3
        t0 = time.perf_counter(); dev = hw.to(self.dev); self._sync()
        return dev, time.perf_counter() - t0

    def matmul(self, x, w, r):
        t = self.torch
        if self.dev.type == "cuda":
            s = t.cuda.Event(enable_timing=True); e = t.cuda.Event(enable_timing=True)
            t.cuda.synchronize(); s.record()
            y = x
            for _ in range(r):
                y = y @ w
            e.record(); t.cuda.synchronize()
            return y, s.elapsed_time(e) / 1e3
        t0 = time.perf_counter(); y = x
        for _ in range(r):
            y = y @ w
        self._sync()
        return y, time.perf_counter() - t0

    def new_activation(self, b, d):
        return self.torch.randn(b, d, device=self.dev, dtype=self.torch.float32)


def make_backend():
    """Auto-detect the best available backend on this Kaggle session."""
    try:
        import torch_xla.core.xla_model  # noqa
        return TorchBackend(want_tpu=True)
    except Exception:
        pass
    try:
        import torch  # noqa
        return TorchBackend(want_tpu=False)
    except Exception:
        return NumpyBackend()


def measure_point(be, d, n_layers, batch, resident_frac, comp_repeats, n_windows):
    n_res = max(0, min(n_layers, round(resident_frac * n_layers)))
    n_off = n_layers - n_res
    resident = [be.new_weight(d) for _ in range(n_res)]
    host_off = [be.new_host_weight(d) for _ in range(n_off)]

    def one_step():
        x = be.new_activation(batch, d); t0 = time.perf_counter(); tx = comp = 0.0
        for hw in host_off:
            dev_w, dt = be.transfer_in(hw); tx += dt
            x, ct = be.matmul(x, dev_w, comp_repeats); comp += ct
        for rw in resident:
            x, ct = be.matmul(x, rw, comp_repeats); comp += ct
        return time.perf_counter() - t0, comp, tx

    one_step()  # discarded warm-up (excludes per-point compilation)
    tots, comps, txs = [], [], []
    for _ in range(n_windows):
        t, c, x = one_step(); tots.append(t); comps.append(c); txs.append(x)
    t_comp = statistics.mean(comps); t_tx = statistics.mean(txs)
    r_c = n_res / n_layers if n_layers else NAN
    r_b = (t_comp / t_tx) if t_tx > 0 else NAN
    return {"r_c": r_c, "t_comp_s": t_comp, "t_transfer_s": t_tx,
            "t_total_s": statistics.mean(tots), "r_b": r_b,
            "t_total_sd": statistics.pstdev(tots)}


def classify(r_c, r_b, tc=0.50, tb=1.0):
    if r_c < tc:
        return "capacity-limited"
    if r_b == r_b and r_b < tb:
        return "io-limited"
    return "coordination-dominated"


def json_safe(v):
    if isinstance(v, float) and not math.isfinite(v):
        return None
    if isinstance(v, dict):
        return {k: json_safe(x) for k, x in v.items()}
    if isinstance(v, list):
        return [json_safe(x) for x in v]
    return v


print("engine loaded")


In [ ]:
# === run the sweeps ==========================================================
be = make_backend()
is_gpu = getattr(be, "device", "").startswith("GPU") or getattr(be, "is_xla", False)

# Auto-scale: full workload on an accelerator, a small one on CPU so it finishes.
if is_gpu:
    D, LAYERS, BATCH, WINDOWS = 2048, 24, 8, 10
    RC_GRID = [0.1, 0.2, 0.3, 0.4, 0.45, 0.5, 0.55, 0.6, 0.75, 0.9, 1.0]
    RB_REPS = [1, 2, 3, 4, 6, 8, 12]
    COMP = 6
else:
    D, LAYERS, BATCH, WINDOWS = 512, 12, 8, 5
    RC_GRID = [0.1, 0.25, 0.4, 0.5, 0.6, 0.75, 1.0]
    RB_REPS = [1, 2, 4, 8]
    COMP = 3

print(f"[measure] backend={be.name} device={be.device} "
      f"d={D} layers={LAYERS} batch={BATCH} windows={WINDOWS}")

summary = []
print("\n=== Sweep A: R_C (residency) ===")
print(f"{'R_C':>6} {'regime':>22} {'T_total(ms)':>13} {'R_B':>8}")
for rc in RC_GRID:
    m = measure_point(be, D, LAYERS, BATCH, rc, COMP, WINDOWS)
    reg = classify(m["r_c"], m["r_b"])
    print(f"{m['r_c']:>6.2f} {reg:>22} {m['t_total_s']*1e3:>13.2f} {m['r_b']:>8.2f}")
    m.update(sweep="R_C", regime=reg, device=be.device); summary.append(m)

print("\n=== Sweep B: R_B (overlap) ===")
print(f"{'reps':>6} {'regime':>22} {'T_total(ms)':>13} {'R_B':>8}")
for rep in RB_REPS:
    m = measure_point(be, D, LAYERS, BATCH, 0.6, rep, WINDOWS)
    reg = classify(m["r_c"], m["r_b"])
    print(f"{rep:>6} {reg:>22} {m['t_total_s']*1e3:>13.2f} {m['r_b']:>8.2f}")
    m.update(sweep="R_B", regime=reg, comp_repeats=rep, device=be.device); summary.append(m)

# --- save (Kaggle output dir) ---
out = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
slug = "".join(c.lower() if c.isalnum() else "-" for c in be.device)
slug = "-".join(filter(None, slug.split("-")))
payload = {"device": be.device, "backend": be.name, "d": D, "n_layers": LAYERS,
           "batch": BATCH, "windows": WINDOWS, "python": platform.python_version(),
           "theta_C": 0.50, "theta_B": 1.0, "points": summary}
fp = out / f"orion_accel_{slug}.json"
fp.write_text(json.dumps(json_safe(payload), indent=2, allow_nan=False))

# --- paper paste-back summary ---
a = [s for s in summary if s["sweep"] == "R_C"]
cap = [s["t_total_s"] for s in a if s["r_c"] < 0.5]
res = [s["t_total_s"] for s in a if s["r_c"] >= 1.0]
print("\n" + "=" * 62)
print("[paper paste-back — send me this block]")
print(f"device = {be.device}")
if cap and res:
    print(f"capacity-limited(R_C<0.5) mean T_total = {statistics.mean(cap)*1e3:.2f} ms")
    print(f"resident(R_C>=1.0)        mean T_total = {statistics.mean(res)*1e3:.2f} ms")
    print(f"capacity crossover        = {statistics.mean(cap)/statistics.mean(res):.2f}x")
print(f"saved: {fp}")
print("=" * 62)


In [ ]:
# === optional: plot the sweeps (matplotlib is preinstalled on Kaggle) ========
try:
    import matplotlib.pyplot as plt
    a = [s for s in summary if s["sweep"] == "R_C"]
    a = sorted(a, key=lambda s: s["r_c"])
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].plot([s["r_c"] for s in a], [s["t_total_s"]*1e3 for s in a], "o-")
    ax[0].axvline(0.5, ls="--", c="r", label=r"$\theta_C=0.5$")
    ax[0].set_xlabel(r"$R_C$ (residency)"); ax[0].set_ylabel("T_total (ms)")
    ax[0].set_title("Sweep A: capacity"); ax[0].legend()
    b = [s for s in summary if s["sweep"] == "R_B" and s["r_b"] == s["r_b"]]
    b = sorted(b, key=lambda s: s["r_b"])
    ax[1].plot([s["r_b"] for s in b], [s["t_total_s"]*1e3 for s in b], "s-")
    ax[1].axvline(1.0, ls="--", c="r", label=r"$\theta_B=1$")
    ax[1].set_xlabel(r"$R_B$ (overlap)"); ax[1].set_ylabel("T_total (ms)")
    ax[1].set_title("Sweep B: overlap"); ax[1].legend()
    plt.tight_layout(); plt.savefig("orion_sweeps.png", dpi=120); plt.show()
    print("saved: orion_sweeps.png")
except Exception as ex:
    print("plot skipped:", ex)
